# 02 · Experiments (the recipe for reproducing the results)

Every experiment and calibration run, **in the order they were run**. Needs a **GPU runtime (T4)** and the dataset;
the setup cell restores the dataset from the archive.

All results are written to Drive under `experiments/<name>/` (per-category metrics, saved anomaly maps, the config
that was actually used). Each run **skips categories that already have saved results**, so after a disconnect you
reconnect, re-run the setup cell and re-run the same cell. Add `--force` to the command to recompute.

| # | Run | Question it answers |
|---|---|---|
| 1 | `padim_r18_256` | Baseline: per-position Gaussian on ResNet-18 features |
| 2 | `patchcore_wrn50_256` | Main method: PatchCore, WideResNet-50, 256 px, 10% coreset |
| 3 | `patchcore_wrn50_384` | Does a higher resolution help small defects? |
| 4 | `patchcore_wrn50_256_cs1` | How much is lost with a 10x smaller memory bank (1% coreset)? |
| 5 | `patchcore_r18_256` | Backbone or method: how much of the PaDiM gap is the backbone? |
| 6 | `patchcore_n{5,10,25,50}_s{0,1,2}` | How many normal images are needed? (3 seeds each) |
| 7 | `patchcore_wrn50_256_final` | Same config as 2, but saves the models (used by calibration, robustness and the app) |
| 8 | held-out calibration (v1) | Normal-only thresholds from a 10% hold-out |
| 9 | 5-fold CV calibration (v2) | More stable thresholds from out-of-fold scores on all normals |
| 10 | robustness | Noise, brightness and blur at test time |

**These are sensitivity analyses, not model selection.** The settings of the main run (layers 2+3, 3x3 local
aggregation, 10% coreset) were fixed in advance, following the PatchCore paper. Nothing was tuned on the test set.

Measured on a T4: the full PatchCore 256 run took about 5 minutes, held-out calibration about 3 minutes,
5-fold CV calibration about 14 minutes, the robustness run 11.3 minutes. The 384 px and few-shot runs were not timed.

After a pull that changes the code, use **Runtime → Restart session** (this keeps `/content` and the dataset).

In [ ]:
import logging, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

NEED_DATA = True   # True: experiments read the images

drive.mount("/content/drive")

REPO_URL = "https://github.com/tsegaeph/industrial-anomaly-detection.git"
REPO_DIR = Path("/content/repo")
DRIVE_PROJECT = Path("/content/drive/MyDrive/industrial-anomaly-detection")
DRIVE_DATA = DRIVE_PROJECT / "datasets" / "mvtec_anomaly_detection"
ARCHIVE = DRIVE_PROJECT / "datasets" / "mvtec_7cat.tar"
DATA_ROOT = Path("/content/industrial-anomaly-detection/datasets/mvtec_anomaly_detection")
EXP = DRIVE_PROJECT / "experiments"
REPORTS = DRIVE_PROJECT / "reports"
CATS = ("bottle", "capsule", "carpet", "metal_nut", "screw", "toothbrush", "zipper")


def sh(*cmd) -> None:
    """Run a command, print its output, raise on failure."""
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.returncode:
        raise RuntimeError(f"{cmd[0]} failed: {r.stderr.strip()}")


# Code always comes from GitHub. After a pull that changes modules you already imported,
# use Runtime -> Restart session (this keeps /content, so the dataset stays).
if REPO_DIR.exists():
    sh("git", "-C", REPO_DIR, "pull")
else:
    sh("git", "clone", REPO_URL, REPO_DIR)
sys.path.insert(0, str(REPO_DIR))


def dataset_ok() -> bool:
    return all((DATA_ROOT / c / "train" / "good").is_dir() for c in CATS)


def ensure_dataset() -> None:
    """Restore the dataset on Colab's local disk if it is missing or incomplete."""
    if dataset_ok():
        print("Dataset already complete on local disk.")
        return
    shutil.rmtree(DATA_ROOT.parent.parent, ignore_errors=True)          # remove any partial copy
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if ARCHIVE.exists():                       # one large file: much faster than thousands of small ones
        sh("cp", ARCHIVE, "/content/mvtec.tar")
        sh("tar", "-xf", "/content/mvtec.tar", "-C", DATA_ROOT.parent)
        Path("/content/mvtec.tar").unlink(missing_ok=True)
    if not dataset_ok():                       # no archive, or extraction failed
        print("Archive unavailable or incomplete, copying file by file (slow)...")
        shutil.rmtree(DATA_ROOT.parent.parent, ignore_errors=True)
        DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(DRIVE_DATA, DATA_ROOT)
    assert dataset_ok(), "Dataset incomplete after copy"
    print("Dataset ready.")


if NEED_DATA:
    ensure_dataset()

logging.basicConfig(level=logging.INFO)

## Helper and metrics self-test

`run(config, name, *overrides)` calls `scripts/run_experiment.py`. With no overrides it runs the config as it is;
with overrides (`key=value`, dotted keys for nested values) it needs a new `name`, so results are never mixed up.

The self-test checks the metrics code on synthetic data. Expected: for `perfect`, all five metrics about 1.0; for
`random`, pixel AUROC about 0.5 and AUPRO about 0.15 (a random map has PRO equal to FPR, so the area under that line
up to FPR 0.3, divided by 0.3, is 0.15). Ignore `image_auroc` on 10 images, it is noise.

In [ ]:
def run(config, name, *overrides, categories=None):
    cmd = (f"python /content/repo/scripts/run_experiment.py --config /content/repo/configs/{config}.yaml "
           f"--data-root {DATA_ROOT} --out-dir {EXP}")
    if categories:
        cmd += " --categories " + " ".join(categories)
    if name != config or overrides:
        cmd += f" --set name={name} " + " ".join(overrides)
    get_ipython().system(cmd)

In [ ]:
import numpy as np
from src.evaluation.metrics import evaluate_category

rng = np.random.default_rng(0)
masks = np.zeros((10, 64, 64), bool); masks[:5, 20:30, 20:30] = True
labels = np.array([1] * 5 + [0] * 5)
perfect = masks.astype(np.float32) + 0.01 * rng.random(masks.shape).astype(np.float32)
noise = rng.random(masks.shape).astype(np.float32)
print("perfect:", evaluate_category(perfect, masks, labels))
print("random: ", evaluate_category(noise, masks, labels))

## 1. Baseline: PaDiM (ResNet-18)

In [ ]:
run("padim_r18_256", "padim_r18_256")

## 2. Main method: PatchCore (WideResNet-50, 256 px)

The first cell is an optional smoke test on one category, so a failure shows up early. The second runs all seven
categories and skips bottle if it already has results.

In [ ]:
run("patchcore_wrn50_256", "patchcore_wrn50_256", categories=["bottle"])

In [ ]:
run("patchcore_wrn50_256", "patchcore_wrn50_256")

## 3. Sensitivity analyses

Resolution 384 (the heaviest run: more patches per image, mean peak GPU memory about 4.4 GB at batch size 8),
1% coreset, and a ResNet-18 backbone (its layers 2+3 have 384 channels, so `target_dim` is 384).

In [ ]:
run("patchcore_wrn50_256", "patchcore_wrn50_384", "image_size=384", "batch_size=8")

In [ ]:
run("patchcore_wrn50_256", "patchcore_wrn50_256_cs1", "params.coreset_ratio=0.01")

In [ ]:
run("patchcore_wrn50_256", "patchcore_r18_256", "params.backbone=resnet18", "params.target_dim=384")

## 4. Few-shot normals

Fit on only 5, 10, 25 or 50 normal images per category (the seed controls which images are sampled); the test set is
always the full one. 12 runs. Toothbrush has only 60 normal training images, so `n=50` is nearly its full set.

In [ ]:
for n in (5, 10, 25, 50):
    for s in (0, 1, 2):
        run("patchcore_wrn50_256", f"patchcore_n{n}_s{s}", f"n_train={n}", f"seed={s}")

## 5. Final models (saved)

Same configuration as run 2, with `save_model=true`. This writes `model_<category>.pt` (the memory banks) next to
the results. Re-running with the same seed reproduced run 2's metrics exactly (to 4 decimals), a reproducibility check.

In [ ]:
run("patchcore_wrn50_256", "patchcore_wrn50_256_final", "save_model=true")

## 6. Decision thresholds from normal data only

AUROC, AP and AUPRO need no threshold; the app's "normal / defective" verdict does. Both calibrations use
training images only and never touch the test set.

- **v1, held-out:** fit on 90% of the normal training images, score the other 10%
  (`calibration_heldout_scores.json`). Only 6 to 32 scores per category.
- **v2, 5-fold cross-validation:** every normal training image is scored by a model that never saw it
  (`calibration_cv_scores.json`), 60 to 320 scores per category. v2 was introduced after v1's test results showed
  unstable thresholds, so it is a documented revision, not a pre-registered rule.

In [ ]:
!python /content/repo/scripts/calibrate_thresholds.py \
  --config /content/repo/configs/patchcore_wrn50_256.yaml \
  --data-root {DATA_ROOT} \
  --out-dir {EXP}/patchcore_wrn50_256_final

In [ ]:
!python /content/repo/scripts/calibrate_cv.py \
  --config /content/repo/configs/patchcore_wrn50_256.yaml \
  --data-root {DATA_ROOT} \
  --out-dir {EXP}/patchcore_wrn50_256_final

## 7. Robustness (test-time corruptions)

The saved models are evaluated on corrupted test images: Gaussian noise (std 10, 25, 50 on the 0-255 scale),
brightness x0.6, 0.8, 1.2, 1.4, and Gaussian blur (sigma 1 and 2 px), applied after resizing to 256 px.
The `clean` row must reproduce the normal results. Known confound: screw and zipper are grayscale replicated to 3
channels, and the noise is added independently per channel, which makes those images coloured. For these two
categories part of the damage may come from colour noise rather than noise as such.

In [ ]:
!python /content/repo/scripts/run_robustness.py \
  --models-dir {EXP}/patchcore_wrn50_256_final \
  --data-root {DATA_ROOT} \
  --out-dir {REPORTS}/metrics/robustness \
  --categories bottle

In [ ]:
import time; t0 = time.time()
!python /content/repo/scripts/run_robustness.py \
  --models-dir {EXP}/patchcore_wrn50_256_final \
  --data-root {DATA_ROOT} \
  --out-dir {REPORTS}/metrics/robustness
print(f"took {(time.time() - t0) / 60:.1f} min")

When the runs are done, go to **Runtime → Disconnect and delete runtime**; everything is saved on Drive. Analysis continues in notebook 03.